# Review refinement boundary dari laporan iPhone

Buka dari root **SongMap**, lalu **Run All**. JSON Who You Say I Am yang Anda kirim sudah dimasukkan ke `experiments/structure/review_inputs/`; notebook langsung membaca file tersebut. Tidak perlu download atau ekspor ulang dari iPhone.

Untuk lagu lain nanti, ganti `REPORTS_DIR` ke folder laporan pilihan, misalnya `Path.home()/'Downloads'`. Gunakan hasil analisis dengan model konteks baru.

Notebook ini memakai standard library Python saja: tidak membutuhkan NumPy, Torch, Core ML, audio, atau inferensi ulang. Ia menilai pergeseran boundary yang sudah ada ke kandidat garis bar di sekitar batas Apple, mempertahankan seluruh label konteks dan jumlah section. Kandidat memerlukan anchor kiri/kanan yang jelas, coverage cukup, serta kenaikan dukungan probabilitas frame. Bukti lemah membuat batas tetap.

Ini eksperimen **review offline**. Kandidat belum diterapkan ke SongMap. Threshold belum dikalibrasi; grid bar aplikasi juga dapat meleset dari ketukan asli. Batas yang hilang, transisi instrumental baru, dan pergeseran outro besar memerlukan evaluasi terpisah.


In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess, sys, json

ROOT=Path.cwd().resolve()
SCRIPT=ROOT/'scripts/review_boundary_shifts.py'
assert SCRIPT.is_file(), 'Buka notebook dari root SongMap'
REPORTS_DIR=ROOT/'experiments/structure/review_inputs'  # JSON Anda sudah disertakan.
# Untuk laporan lain: REPORTS_DIR=Path.home()/'Downloads'
REFERENCE_CSV=None  # Opsional setelah mengisi reference_seconds pada CSV hasil review.
assert REPORTS_DIR.is_dir(), f'Folder tidak ditemukan: {REPORTS_DIR}'
assert list(REPORTS_DIR.glob('analysis-*.json')), 'Tidak ada analysis-*.json di REPORTS_DIR'
OUTPUT=ROOT/'outputs.local'/('boundary-shifts-'+datetime.now().strftime('%Y%m%d-%H%M%S-%f'))
print('Sumber:',REPORTS_DIR)
print('Hasil:',OUTPUT)


In [ ]:
command=[sys.executable,'-u',str(SCRIPT),'--reports-dir',str(REPORTS_DIR),'--output-dir',str(OUTPUT)]
if REFERENCE_CSV is not None:
    command+=['--reference-csv',str(REFERENCE_CSV)]
log=Path(str(OUTPUT)+'.log');log.parent.mkdir(parents=True,exist_ok=True)
with log.open('w',encoding='utf-8') as handle:
    process=subprocess.Popen(command,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in process.stdout:
            print(line,end='');handle.write(line);handle.flush()
        status=process.wait()
    except KeyboardInterrupt:
        process.terminate();process.wait();raise
if status:
    raise RuntimeError(f'Gagal ({status}); lihat {log}')


In [ ]:
from IPython.display import display, Markdown
import html

review=json.loads((OUTPUT/'review.json').read_text())
def clock(seconds):
    return f'{int(seconds//60):02d}:{seconds%60:05.2f}'
lines=['| Lagu | Transisi | Batas lama | Kandidat | Pergeseran |',
       '|---|---|---|---|---|']
for track in review['tracks']:
    for row in track['boundaries']:
        if row['status']!='candidate_requires_review':
            continue
        title=html.escape(track['title']).replace('|',' / ')
        lines.append(f"| {title} | {row['left_label']} → {row['right_label']} | {clock(row['original_seconds'])} | {clock(row['candidate_seconds'])} | {row['candidate_seconds']-row['original_seconds']:+.2f}s |")
print('Reports:',review['reports'],'| judul unik:',review['unique_titles'],'| kandidat:',review['candidates'])
print('Laporan lama/belum lengkap yang dilewati:',len(review['skipped_reports']))
if review['candidates']:
    display(Markdown('\n'.join(lines)))
else:
    print('Tidak ada pergeseran dengan dukungan cukup pada aturan review ini.')
if 'manual_spot_check' in review:
    print(json.dumps(review['manual_spot_check'],indent=2))
print('Kirim:',OUTPUT/'review.json')
print('CSV opsional:',OUTPUT/'boundary_review.csv')
print('Belum mengubah peta lagu, label manual, atau model aplikasi.')


Untuk contoh **Who You Say I Am**, periksa bar sekitar **04:41.30–04:43.39** sambil mendengarkan. Kandidat waktu bukan label manual baru.

Untuk menilai pada beberapa lagu, ekspor hasil analisis konteks lagu lain ke folder yang sama, lalu Run All kembali. Laporan dengan konteks tidak lengkap dilewati; bukti yang rusak ditolak. Beberapa laporan judul sama adalah pengulangan pengukuran, bukan lagu independen baru.

`boundary_review.csv` memuat boundary antar-label yang berbeda, termasuk yang tetap/tidak memiliki kandidat. Anda **tidak perlu melabeli ulang semua section**. Jika ingin mencatat pemeriksaan batas tertentu, isi hanya `reference_seconds` dengan detik waktu perpindahan yang Anda dengar (contoh format `283.4`) dan `review_note`; biarkan baris yang belum didengar kosong. Jangan ubah kolom identitas, label atau waktu kandidat. Jangan menyalin kandidat menjadi referensi tanpa mendengarkan.

Set `REFERENCE_CSV` ke path CSV yang sudah Anda isi lalu Run All. Perbandingan error waktu hanya dihitung pada referensi yang Anda isi dan terikat hash laporan asal; ia tetap bukan ukuran akurasi boundary seluruh dataset. Gunakan kumpulan laporan asal yang sama saat membaca CSV lama.

Kirim **review.json** terlebih dahulu. Hasil review ini tidak mengaktifkan fitur baru atau mengganti resource di Xcode.
